In [1]:
import site
print(site.getsitepackages())

['/workspaces/llm-tutorial-homework/02-vector-search/llm-zoomcamp-hw2/.venv/lib/python3.12/site-packages']


In [2]:
import sys
print(sys.executable)

/workspaces/llm-tutorial-homework/02-vector-search/llm-zoomcamp-hw2/.venv/bin/python3


In [3]:
import embedder

2026-06-30 11:41:04.521041209 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [14]:
import numpy as np
from embedder import Embedder

In [15]:
text='How does approximate nearest neighbor search work?'

In [16]:
em=Embedder()

In [17]:
embedding=em.encode(text)

In [18]:
#print(embedding)
print("Embedding shape:", embedding.shape)

Embedding shape: (384,)


In [19]:
embedding[0]

np.float64(-0.02058203437252893)

Now we're ready to do the homework.

Q1. Embedding a query  
Embed the following query:  

How does approximate nearest neighbor search work?  

The embedder returns a vector of 384 numbers. What's the first value (v[0])?  

-0.31  
**-0.02**  
0.12  
0.44  

In [20]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [21]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [22]:
for i in documents:
    if i['filename']=='02-vector-search/lessons/07-sqlitesearch-vector.md':
        filecontent=i['content']
 

In [23]:
v1=em.encode(filecontent)

In [24]:
embedding.dot(v1)

np.float64(0.36107026789538205)

## Q2. Cosine similarity  
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.  

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
  
0.07  
**0.37**  
0.68  
0.92  



## Q3. Chunking and search by hand  
#### A full page covers several topics, which waters down its embedding.  

We chunk the pages the same way as in homework 1:  



In [25]:

from gitsource import chunk_documents  
chunks = chunk_documents(documents, size=2000, step=1000)  


In [26]:
len(chunks)
chunks[10]

{'start': 4000,
 'content': "hat the LLM generates.\nThat search step is what gives the LLM the context it needs to answer\ncorrectly.\n\nWhat we just did was naive. I knew in advance which FAQ entry held the\nanswer and pasted it in by hand. What we want instead is to perform\nsearch automatically. We take the student's question, find the most\nrelevant documents, and send those to the LLM.\n\nIn code, it looks like this:\n\n```python\ndef rag(question):\n    search_results = search(question)\n    user_prompt = build_prompt(question, search_results)\n    return llm(user_prompt)\n```\n\nThat's the entire architecture. It comes down to three components.\n\nThe pieces are search, the prompt, and the LLM:\n\n- search\n- prompt\n- LLM\n\n\n```mermaid\nflowchart TD\n    U([User])\n\n    APP[Application]\n\n    DB[(DB)]\n    DOCS[[D1 ... D5]]\n\n    PROMPT[Build Prompt<br/>Question + Context]\n\n    LLM[LLM]\n\n    ANSWER([Answer])\n\n    U -->|Question| APP\n\n    APP -->|Query| DB\n    DB 

### We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:  



In [27]:
# Extract the text field from each dictionary
text_list = [chunk['content'] for chunk in chunks]

# Pass the clean list of strings
vectors=em.encode_batch(text_list)

In [28]:
vectors.shape , len(text_list)

((295, 384), 295)

In [29]:
X=np.array(vectors)
X.shape

(295, 384)

In [30]:
q1='How does approximate nearest neighbor search work?'
q1v=em.encode(q1)
q1v.shape

(384,)

In [31]:
scores=X.dot(q1v)
scores.shape

(295,)

In [32]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(94), np.float64(0.6489016436447387))

In [33]:
chunks[94]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

## Q3 Which file does the highest-scoring chunk belong to (its filename)?  

02-vector-search/lessons/03-embeddings-dataset.md  
02-vector-search/lessons/06-rag-vector.md  
**02-vector-search/lessons/07-sqlitesearch-vector.md**  
02-vector-search/lessons/09-onnx-embedder.md  

## Question 4

In [37]:
X.shape, len(documents)

((295, 384), 72)

In [45]:
documents[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

In [39]:
texts = []

for doc in documents:
    text = doc['content'] + ' ' + doc['filename']
    texts.append(text)

In [46]:
len(texts), len(documents)

(72, 72)

In [52]:
#texts[1]

In [50]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = em.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/2 [00:00<?, ?it/s]

72

In [53]:
X=np.array(vectors)
X.shape

(72, 384)

In [54]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['content'])
vindex.fit(X, documents)

In [70]:
q1='What metric do we use to evaluate a search engine?'
v1=em.encode(q1)
results=vindex.search(v1, num_results=5)
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q4. Vector search with minsearch  
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.  

Let's use VectorSearch from minsearch and run a search for the following query:  

What metric do we use to evaluate a search engine?  

Which file is the filename of the first result?  

02-vector-search/lessons/04-vector-search.md  
**04-evaluation/lessons/05-search-metrics.md**  
04-evaluation/lessons/13-llm-as-judge.md  
05-monitoring/lessons/04-metrics.md  

## Q5. Text search vs vector search  
Vector search matches by meaning, keyword search by exact words.  

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.  

Run both searches for this query:  

How do I store vectors in PostgreSQL? 

In [71]:
from minsearch import Index

index = Index(
    text_fields=['content'],
     keyword_fields=["filename"]
    
)
index.fit(documents)

In [72]:
question="How do I store vectors in PostgreSQL?"

In [74]:
keyword_search_results = index.search(
    question,
    boost_dict={'content': 1.0},
  
    num_results=5
)

for i in keyword_search_results:
    print(i['filename'])

02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/03-embeddings-dataset.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/10-next-steps.md


In [69]:
v1=em.encode(question)
results=vindex.search(v1, num_results=5)
for i in results:
    print(i['filename'])

02-vector-search/lessons/08-pgvector.md
05-monitoring/lessons/05-database.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/04-vector-search.md
02-vector-search/lessons/07-sqlitesearch-vector.md


In [76]:
query_vec = em.encode(question)

vector_scores = X.dot(query_vec)
top_idx = vector_scores.argsort()[-5:][::-1]

vector_results = [chunks[i] for i in top_idx]

In [79]:
for i in vector_results:
    print(i['filename'])

01-agentic-rag/lessons/06-building-prompt.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/06-building-prompt.md


In [81]:
keyword_files = {r["filename"] for r in keyword_search_results}
vector_files = {r["filename"] for r in vector_results}

vector_files - keyword_files

{'01-agentic-rag/lessons/05-search.md',
 '01-agentic-rag/lessons/06-building-prompt.md',
 '01-agentic-rag/lessons/13-function-calling.md'}

 

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?  

02-vector-search/lessons/01-intro.md  
02-vector-search/lessons/02-embeddings.md  
02-vector-search/lessons/08-pgvector.md  
03-orchestration/lessons/05-rag.md  

## Q6. Hybrid search  
Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))
"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

A document that ranks well in both lists ends up higher than one that's only strong in a single list.

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]
Now run the query "How do I give the model access to tools?" with vector and text search and fuse the results with rrf:

results = rrf([vector_results, text_results])
Which file is ranked first after RRF?

01-agentic-rag/lessons/01-intro.md  
01-agentic-rag/lessons/13-function-calling.md  
01-agentic-rag/lessons/14-agentic-loop.md  
01-agentic-rag/lessons/16-other-frameworks.md  
Notice that this file isn't first in either search on its own - it wins because it ranks high in both.  